# Rubin/LSST — Stable Stars with Known Spectral Type: Rubin Visit Cross-Match

For each photometrically stable star with a **known spectral type** (from the
Simbad master catalogue built in notebook `01_findStarsinSimbad.ipynb`),
this notebook:

1. Loads the master Simbad catalogue and **filters to stars with a defined spectral type**.
2. Loads the Rubin visit table (`visitTable-..._WithTracts.parquet`).
3. **Cross-matches** each star against visits using a cone-search on the visit
   pointing (RA, Dec) — visit footprint approximated by Rubin FoV radius ≈ 1.75°.
4. Saves per-star visit lists (CSV + Parquet) in `data_SIMBAD_02/per_star/`.
5. Produces a **summary table** (visit counts per star × band, sorted by total
   visits descending).


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-06-22
- last update : 2026-06-23 : add airmass vs mjd curves 
- input catalogue : `data_SIMBAD_01/master_stable_stars_V17-22_r1.5deg.csv`
- input visits : `../05_runbindata_visits/data_fromlsst/visitTable-2025041500138-2026053000760_N83426_WithTracts.parquet`


## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from astropy.coordinates import SkyCoord
import astropy.units as u

warnings.filterwarnings("ignore")

print(f"numpy   version : {np.__version__}")
print(f"pandas  version : {pd.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → inline")

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────────────
# Simbad master catalogue from notebook 01
PATH_SIMBAD_MASTER = "data_SIMBAD_01/master_stable_stars_V17-20_r1.5deg.csv"

# Rubin visit table (WithTracts parquet)
# PATH_VISITS = "./rubindata_visits/visitTable-2025041500138-2026053000760_N83426_WithTracts.parquet"
PATH_VISITS = "./rubindata_visits/visitTable-2025041500138-2026061900757_N88665_WithTracts.parquet"

# ── Cross-match radius ─────────────────────────────────────────────────────────────
# Rubin field of view radius ~ 1.75 deg (diameter 3.5 deg).
# We match a star to a visit if the angular distance between the star and
# the visit pointing is smaller than MATCH_RADIUS_DEG.
MATCH_RADIUS_DEG = 1.75  # [adjustable]

# ── Bands to keep ────────────────────────────────────────────────────────────────
BANDS_TO_KEEP = ["u", "g", "r", "i", "z", "y"]  # exclude 'ph', 'unknown'

# ── Output directories ──────────────────────────────────────────────────────────────
NB_TAG = "SIMBAD_02"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
DIR_STARS = os.path.join(DIR_DATA, "per_star")  # one file per star
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
os.makedirs(DIR_STARS, exist_ok=True)
print(f"Data root : {os.path.abspath(DIR_DATA)}")
print(f"Per-star  : {os.path.abspath(DIR_STARS)}")
print(f"Figs      : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ───────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{chr(123)}pdf,png{chr(125)}")


print("Configuration done.")

## 2. Load the Simbad master catalogue

We load the master catalogue produced by `01_findStarsinSimbad.ipynb` and
**retain only stars whose spectral type is defined** (non-empty, not `?`, not `unknown`).


In [ ]:
df_simbad = pd.read_csv(PATH_SIMBAD_MASTER)
print(f"Master catalogue shape  : {df_simbad.shape}")
print(f"Columns                 : {list(df_simbad.columns)}")
df_simbad.head(3)

In [ ]:
pd.set_option("display.max_rows", 50)

In [ ]:
def has_known_spectral_type(sp) -> bool:
    """Return True when sp is a meaningful spectral-type string."""
    if pd.isna(sp):
        return False
    s = str(sp).strip()
    return s not in ("", "?", "unknown", "--", "nan", "None")


mask_sp = df_simbad["spectral_type"].apply(has_known_spectral_type)
df_stars = df_simbad[mask_sp].copy().reset_index(drop=True)

print(f"Stars with known spectral type : {len(df_stars)} / {len(df_simbad)}")
print("Spectral-type values (top-15)  :")
print(df_stars["spectral_type"].value_counts().head(15).to_string())
df_stars[["simbad_id", "ra_deg", "dec_deg", "V_mag", "spectral_type", "field"]]

In [ ]:
print("Breakdown by DDF field:")
print(df_stars["field"].value_counts().to_string())

In [ ]:
# ── Extract simple MK temperature class from the first character of spectral_type
def mk_class_from_sptype(sp: str) -> str:
    s = str(sp).strip().upper()
    for cls in ("O", "B", "A", "F", "G", "K", "M", "L", "T", "Y", "W", "C", "S"):
        if s.startswith(cls):
            return cls
    return "?"


df_stars["mk_class_simple"] = df_stars["spectral_type"].apply(mk_class_from_sptype)
print("MK class distribution:")
print(df_stars["mk_class_simple"].value_counts().to_string())

## 3. Load the Rubin visit table

We load the full visit table and **pre-filter to science-quality bands**
(u, g, r, i, z, y) to discard engineering (`ph`, `unknown`) entries.
We also compute the **airmass** from the zenith angle: X = 1 / cos(zenith_angle).


In [ ]:
df_visits_raw = pd.read_parquet(PATH_VISITS)
print(f"Visit table raw shape : {df_visits_raw.shape}")
print(f"Bands present         : {sorted(df_visits_raw['band'].unique())}")
df_visits_raw[["id", "band", "ra", "dec", "zenith_angle", "mjd", "tract", "patch", "target"]].head(3)

In [ ]:
# ── Keep only science bands
df_visits = df_visits_raw[df_visits_raw["band"].isin(BANDS_TO_KEEP)].copy()
df_visits.reset_index(drop=True, inplace=True)

# ── Airmass: plane-parallel approximation X = sec(z) = 1/cos(z_rad)
df_visits["airmass"] = 1.0 / np.cos(np.radians(df_visits["zenith_angle"]))

# ── Rename id → visitId
df_visits.rename(columns={"id": "visitId"}, inplace=True)

# ── Select columns of interest
VISIT_COLS = [
    "visitId",
    "band",
    "ra",
    "dec",
    "mjd",
    "airmass",
    "zenith_angle",
    "tract",
    "patch",
    "patch_str",
    "day_obs",
    "seq_num",
    "time_start",
    "target",
]
df_visits = df_visits[VISIT_COLS]

print(f"Visit table filtered shape : {df_visits.shape}")
print(f"Bands kept                 : {sorted(df_visits['band'].unique())}")
print(f"Airmass range              : [{df_visits['airmass'].min():.3f}, {df_visits['airmass'].max():.3f}]")
df_visits.head(3)

## 4. Build SkyCoord arrays for fast angular cross-match

We use `astropy.coordinates.SkyCoord.separation()` to compute the angular distance
between each star and all visit pointings. The visit SkyCoord array is built **once**
before the star loop for efficiency.


In [ ]:
# SkyCoord for ALL visits (built once)
visit_coords = SkyCoord(
    ra=df_visits["ra"].values * u.deg,
    dec=df_visits["dec"].values * u.deg,
    frame="icrs",
)
print(f"Visit SkyCoord array : {len(visit_coords)} entries")

# SkyCoord for ALL stars (used for sky-distribution plot later)
star_coords = SkyCoord(
    ra=df_stars["ra_deg"].values * u.deg,
    dec=df_stars["dec_deg"].values * u.deg,
    frame="icrs",
)
print(f"Star SkyCoord array  : {len(star_coords)} entries")

## 5. Per-star visit cross-match and saving

For each star we:
- Select visits within `MATCH_RADIUS_DEG` of the star's sky position.
- Annotate the sub-table with star metadata (`simbad_id`, `spectral_type`, `V_mag`, `field`).
- Save CSV + Parquet in `data_SIMBAD_02/per_star/<star_id>.{csv,parquet}`.
- Accumulate per-band counts for the summary table.


In [ ]:
match_radius = MATCH_RADIUS_DEG * u.deg
summary_rows = []

for idx, star in df_stars.iterrows():
    star_coord = SkyCoord(ra=star["ra_deg"] * u.deg, dec=star["dec_deg"] * u.deg, frame="icrs")

    sep = star_coord.separation(visit_coords)
    in_fov = sep < match_radius
    df_match = df_visits[in_fov].copy()

    row = {
        "simbad_id": star["simbad_id"],
        "spectral_type": star["spectral_type"],
        "mk_class_simple": star.get("mk_class_simple", "?"),
        "V_mag": star["V_mag"],
        "ra_deg": star["ra_deg"],
        "dec_deg": star["dec_deg"],
        "field": star["field"],
        "n_visits_total": len(df_match),
    }
    for b in BANDS_TO_KEEP:
        row[f"n_{b}"] = 0

    if not df_match.empty:
        df_match["simbad_id"] = star["simbad_id"]
        df_match["spectral_type"] = star["spectral_type"]
        df_match["V_mag"] = star["V_mag"]
        df_match["field"] = star["field"]

        star_id_clean = star["simbad_id"].replace(" ", "_").replace("/", "-").replace(":", "-")

        df_match.to_csv(os.path.join(DIR_STARS, f"{star_id_clean}.csv"), index=False)
        df_match.to_parquet(os.path.join(DIR_STARS, f"{star_id_clean}.parquet"), index=False)

        band_counts = df_match["band"].value_counts()
        for b in BANDS_TO_KEEP:
            row[f"n_{b}"] = int(band_counts.get(b, 0))

    summary_rows.append(row)

    if (idx + 1) % 20 == 0 or (idx + 1) == len(df_stars):
        print(f"  [{idx + 1:4d}/{len(df_stars)}] {star['simbad_id']:40s}  → {len(df_match):5d} visits")

print(f"\nDone. Per-star files saved to: {os.path.abspath(DIR_STARS)}")

## 6. Summary table — visit counts per star × band

One row per star, sorted by `n_visits_total` descending.
Saved as CSV and Parquet in `data_SIMBAD_02/`.


In [ ]:
df_summary = pd.DataFrame(summary_rows)
df_summary.sort_values("n_visits_total", ascending=False, inplace=True)
df_summary.reset_index(drop=True, inplace=True)

col_order = [
    "simbad_id",
    "spectral_type",
    "mk_class_simple",
    "V_mag",
    "ra_deg",
    "dec_deg",
    "field",
    "n_visits_total",
] + [f"n_{b}" for b in BANDS_TO_KEEP]
df_summary = df_summary[col_order]

print(f"Summary table shape : {df_summary.shape}")
df_summary.head(15)

In [ ]:
out_csv = os.path.join(DIR_DATA, "summary_visit_counts_per_star.csv")
out_parquet = os.path.join(DIR_DATA, "summary_visit_counts_per_star.parquet")
df_summary.to_csv(out_csv, index=False)
df_summary.to_parquet(out_parquet, index=False)
print(f"Saved: {out_csv}")
print(f"Saved: {out_parquet}")

In [ ]:
print("=== Visit-count statistics (total visits per star) ===")
print(df_summary["n_visits_total"].describe().to_string())
print()
print("=== Stars with >= 100 visits ===")
print(
    df_summary[df_summary["n_visits_total"] >= 100][
        ["simbad_id", "spectral_type", "field", "n_visits_total"]
    ].to_string()
)

## 7. Visualisations

### 7.1  MK spectral-class composition per DDF (stacked barplot)

One bar per DDF field; each bar is colour-stacked by MK temperature class (O/B/A/F/G/K/M/…).
This representation reveals minority spectral types that are invisible in overlapping histograms.


In [ ]:
# ── Palette: one colour per MK class, in temperature order O→M + unknowns ────────
MK_ORDER = ["O", "B", "A", "F", "G", "K", "M", "L", "T", "W", "C", "S", "?"]
MK_COLORS = {
    "O": "#6699FF",  # blue-violet
    "B": "#99CCFF",  # light blue
    "A": "#DDDDDD",  # light grey  (white stars → use grey for visibility)
    "F": "#FFFFAA",  # pale yellow
    "G": "#FFDD00",  # yellow  (Sun-like)
    "K": "#FF8800",  # orange
    "M": "#EE2222",  # red
    "L": "#993300",  # brown
    "T": "#663300",  # dark brown
    "W": "#CC33FF",  # violet (Wolf-Rayet)
    "C": "#009966",  # green (carbon)
    "S": "#336666",  # teal
    "?": "#AAAAAA",  # grey (unclassified)
}

# ── Cross-tabulation: rows = DDF field, columns = MK class ──────────────────────
ct = df_summary.groupby(["field", "mk_class_simple"]).size().unstack(fill_value=0)
mk_present = [c for c in MK_ORDER if c in ct.columns]
ct = ct[mk_present]

fields = ct.index.tolist()
x = np.arange(len(fields))
bar_w = 0.6

fig, ax = plt.subplots(figsize=(max(6, len(fields) * 1.5), 5))
bottom = np.zeros(len(fields))
for cls in mk_present:
    vals = ct[cls].values.astype(float)
    color = MK_COLORS.get(cls, "#CCCCCC")
    ec = "#555555" if color in ("#DDDDDD", "#FFFFAA", "#FFDD00") else "none"
    ax.bar(x, vals, bar_w, bottom=bottom, label=cls, color=color, edgecolor=ec, linewidth=0.4)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(fields, fontsize=10, rotation=15, ha="right")
ax.set_ylabel("N stable stars with known spectral type")
ax.set_title("MK spectral-class composition of stable stars per DDF")
ax.legend(title="MK class", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9, framealpha=0.8)
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
savefig("barplot_mk_class_per_ddf")
plt.show()

### 7.2  N visits per star per DDF — stacked by band (subplots)

One subplot per DDF field.
X axis = Simbad star name (sorted by total visits descending).
Y axis = N visits, colour-stacked in band order u → g → r → i → z → y.


In [ ]:
# ── Band colour palette (wavelength order u=violet … y=near-IR) ─────────────────
BAND_COLORS = {
    "u": "#7B2FBE",  # violet
    "g": "#1A9850",  # green
    "r": "#D73027",  # red
    "i": "#F46D43",  # orange
    "z": "#8B0000",  # dark red
    "y": "#4D1B00",  # near-IR brown
}

fields_ordered = sorted(df_summary["field"].unique())
n_fields = len(fields_ordered)

# ── Layout: 2 columns ────────────────────────────────────────────────────────────
ncols = min(2, n_fields)
nrows = int(np.ceil(n_fields / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 9, nrows * 5), constrained_layout=True)
axes_flat = np.array(axes).flatten()

for ax_idx, field in enumerate(fields_ordered):
    ax = axes_flat[ax_idx]
    sub = (
        df_summary[df_summary["field"] == field]
        .sort_values("n_visits_total", ascending=False)
        .reset_index(drop=True)
    )

    if sub.empty:
        ax.set_visible(False)
        continue

    x_pos = np.arange(len(sub))
    bar_w = 0.75
    bottom = np.zeros(len(sub))

    for band in BANDS_TO_KEEP:
        vals = sub[f"n_{band}"].values.astype(float)
        ax.bar(x_pos, vals, bar_w, bottom=bottom, label=band, color=BAND_COLORS[band])
        bottom += vals

    # X-tick labels: font size adapts to the number of stars
    xlabels = sub["simbad_id"].tolist()
    tick_fs = max(4, min(8, 90 // max(len(xlabels), 1)))
    ax.set_xticks(x_pos)
    ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=tick_fs)

    ax.set_ylabel("N visits")
    ax.set_title(f"DDF: {field}  ({len(sub)} stars)", fontsize=10, fontweight="bold")
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)
    ax.legend(title="band", ncol=6, fontsize=7, loc="upper right", framealpha=0.7)

# Hide unused subplots
for ax_idx in range(n_fields, len(axes_flat)):
    axes_flat[ax_idx].set_visible(False)

fig.suptitle(
    "N Rubin visits per stable star (known spectral type)\nstacked by band  [u g r i z y]",
    fontsize=12,
    fontweight="bold",
)
savefig("barplot_visits_per_star_per_ddf")
plt.show()

### 7.3  Total visits distribution by MK spectral class

In [ ]:
mk_classes = sorted(df_summary["mk_class_simple"].unique())
cmap_mk = cm.get_cmap("tab10", len(mk_classes))

fig, ax = plt.subplots(figsize=(8, 4))
for i, cls in enumerate(mk_classes):
    sub = df_summary[df_summary["mk_class_simple"] == cls]["n_visits_total"]
    if sub.empty:
        continue
    ax.hist(sub, bins=30, alpha=0.55, label=f"{cls} ({len(sub)})", color=cmap_mk(i))
ax.set_xlabel("N visits total")
ax.set_ylabel("N stars")
ax.set_title("Total visit count distribution by MK class")
ax.legend(fontsize=8, ncol=3)
plt.tight_layout()
savefig("hist_visits_by_mk_class")
plt.show()

### 7.4  Sky distribution coloured by total visit count

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(
    df_summary["ra_deg"],
    df_summary["dec_deg"],
    c=df_summary["n_visits_total"],
    cmap="plasma",
    s=20,
    alpha=0.8,
    edgecolors="none",
)
plt.colorbar(sc, ax=ax, label="N visits total")
ax.set_xlabel("RA (deg)")
ax.set_ylabel("Dec (deg)")
ax.set_title("Sky distribution of stable stars with known spectral type\n(colour = N Rubin visits)")
ax.invert_xaxis()
plt.tight_layout()
savefig("sky_distribution_visit_count")
plt.show()

## 8. Example: visit list for the best-observed star

Display the full visit list for the star with the highest total visit count,
and plot its MJD timeline coloured by band.


In [ ]:
best_star = df_summary.iloc[0]
print("Best-observed star:")
print(f"  simbad_id     : {best_star['simbad_id']}")
print(f"  spectral_type : {best_star['spectral_type']}")
print(f"  V_mag         : {best_star['V_mag']:.2f}")
print(f"  field         : {best_star['field']}")
print(f"  n_visits_total: {best_star['n_visits_total']}")
for b in BANDS_TO_KEEP:
    print(f"    n_{b}: {int(best_star[f'n_{b}'])}")

In [ ]:
star_id_clean = best_star["simbad_id"].replace(" ", "_").replace("/", "-").replace(":", "-")
path_best = os.path.join(DIR_STARS, f"{star_id_clean}.parquet")

df_best = pd.read_parquet(path_best)
print(f"Shape of visit list: {df_best.shape}")
df_best[["visitId", "band", "mjd", "airmass", "ra", "dec", "tract", "patch_str", "time_start"]].head(10)

In [ ]:
band_colors = {
    "u": "navy",
    "g": "forestgreen",
    "r": "crimson",
    "i": "darkorange",
    "z": "purple",
    "y": "saddlebrown",
}

fig, ax = plt.subplots(figsize=(12, 3))
for band, grp in df_best.groupby("band"):
    ax.scatter(
        grp["mjd"],
        grp["airmass"],
        label=band,
        color=band_colors.get(band, "grey"),
        s=15,
        alpha=0.7,
        edgecolors="none",
    )
ax.set_xlabel("MJD")
ax.set_ylabel("Airmass")
ax.set_title(
    f"Visit timeline for {best_star['simbad_id']}  [{best_star['spectral_type']}]  V={best_star['V_mag']:.1f}"
)
ax.legend(title="band", ncol=6, fontsize=8)
plt.tight_layout()
savefig(f"timeline_{star_id_clean}")
plt.show()

## 9. Printed summary table (top 30 stars)

Final pretty-printed summary sorted by total visit count.


In [ ]:
cols_display = ["simbad_id", "spectral_type", "mk_class_simple", "V_mag", "field", "n_visits_total"] + [
    f"n_{b}" for b in BANDS_TO_KEEP
]

pd.set_option("display.max_colwidth", 35)
pd.set_option("display.max_rows", 30)
df_summary[cols_display].head(30)

In [ ]:
print("=" * 62)
print("Summary")
print("=" * 62)
print(f"  Total stable stars with known spectral type : {len(df_stars)}")
print(f"  Stars with >= 1 Rubin visit                : {(df_summary['n_visits_total'] > 0).sum()}")
print(f"  Stars with   0 Rubin visits                : {(df_summary['n_visits_total'] == 0).sum()}")
print(f"  Per-star files saved in                    : {os.path.abspath(DIR_STARS)}")
print(f"  Summary table saved in                     : {os.path.abspath(DIR_DATA)}")

## 5. Airmass vs MJD for each stable star

For every star in `df_stars` (stable, known spectral type, 17 ≤ V ≤ 20),
we load its per-star visit CSV and call `plot_airmass_vs_mjd()` which:

- Plots **airmass vs MJD** with a distinct colour per band (u, g, r, i, z, y).
- Adds a **secondary x-axis** on top showing the UTC date (YYYY-MM-DD).
- One figure per star, saved to `figs_SIMBAD_02/`.


In [ ]:
# ── Band colour palette (ordered u → y) ──────────────────────────────────────────────
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]
BAND_COLORS = {
    "u": "#7B2FBE",  # violet
    "g": "#3A86FF",  # blue
    "r": "#06D6A0",  # green
    "i": "#FFB703",  # amber
    "z": "#FB5607",  # orange
    "y": "#E63946",  # red
}


def plot_airmass_vs_mjd(
    df_star: "pd.DataFrame",
    simbad_id: str,
    spectral_type: str,
    v_mag: float,
    field: str,
    save: bool = True,
) -> None:
    """Plot airmass vs MJD for one stable star, one colour per band.

    Parameters
    ----------
    df_star : DataFrame
        Per-star visit table with at least columns ``mjd``, ``band``,
        ``airmass``, and ``time_start``.
    simbad_id : str
        Simbad identifier used for the figure title and filename.
    spectral_type : str
        MK spectral type (e.g. ``'K7'``, ``'M4V'``).
    v_mag : float
        V-band magnitude from Simbad.
    field : str
        DDF name (e.g. ``'COSMOS'``).
    save : bool
        If True, save the figure as PDF + PNG via ``savefig()``.
    """
    from astropy.time import Time

    fig, ax = plt.subplots(figsize=(12, 3))

    # Sort visits by MJD for clean scatter layout
    df_plot = df_star.sort_values("mjd")

    # Plot each band in canonical order u → y
    for band in BAND_ORDER:
        sub = df_plot[df_plot["band"] == band]
        if sub.empty:
            continue
        ax.scatter(
            sub["mjd"],
            sub["airmass"],
            s=10,
            color=BAND_COLORS[band],
            label=band,
            alpha=0.7,
            linewidths=0,
        )

    # ── Secondary x-axis on top: date labels (YYYY-MM-DD) ──────────────────────────
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())

    # Evenly-spaced MJD ticks converted to ISO date strings
    mjd_min, mjd_max = df_plot["mjd"].min(), df_plot["mjd"].max()
    mjd_ticks = np.linspace(mjd_min, mjd_max, 8)
    date_labels = [Time(m, format="mjd").to_datetime().strftime("%Y-%m-%d") for m in mjd_ticks]
    ax_top.set_xticks(mjd_ticks)
    ax_top.set_xticklabels(date_labels, rotation=30, ha="left", fontsize=7)
    ax_top.set_xlabel("Date (UTC)", fontsize=8)

    # ── Axis labels, title, legend ─────────────────────────────────────────────────────────
    ax.set_xlabel("MJD")
    ax.set_ylabel("Airmass")
    ax.invert_yaxis()
    ax.set_title(
        f"{simbad_id}  |  SpT: {spectral_type}  |  V = {v_mag:.2f}  |  field: {field}",
        fontsize=9,
    )
    ax.legend(
        title="band",
        loc="upper right",
        fontsize=10,
        markerscale=3,
        framealpha=0.6,
    )

    plt.tight_layout()

    if save:
        # Build a filesystem-safe filename from the Simbad ID
        safe_id = simbad_id.replace(" ", "_").replace("/", "-")
        savefig(f"airmass_vs_mjd_{safe_id}")

    plt.show()
    # plt.close(fig)


print("plot_airmass_vs_mjd() defined.")

In [ ]:
# ── Loop: airmass vs MJD for every stable star (17 ≤ V ≤ 20, known SpT) ─────────────
# df_stars is expected to be already filtered:
#   - stable == True
#   - spectral_type defined (non-empty, not '?', not 'unknown')
#   - 17 <= V_mag <= 20
# Per-star CSV files live in DIR_STARS as <safe_id>.csv.

n_stars = len(df_stars)
n_plotted = 0
n_missing = 0

print(f"Plotting airmass vs MJD for {n_stars} stable stars (17 ≤ V ≤ 20, known SpT)...")
print("=" * 70)

for _, row in df_stars.iterrows():
    simbad_id = row["simbad_id"]
    spectral_type = row["spectral_type"]
    v_mag = row["V_mag"]
    field = row["field"]

    # Reconstruct the per-star CSV path (same naming convention as the cross-match loop)
    safe_id = simbad_id.replace(" ", "_").replace("/", "-")
    star_file = os.path.join(DIR_STARS, f"{safe_id}.csv")

    if not os.path.exists(star_file):
        print(f"  [SKIP] {simbad_id} — per-star file not found: {star_file}")
        n_missing += 1
        continue

    # Load per-star visit table and restrict to canonical bands
    df_star = pd.read_csv(star_file)
    df_star = df_star[df_star["band"].isin(BAND_ORDER)].copy()

    if df_star.empty:
        print(f"  [SKIP] {simbad_id} — no visits in canonical bands.")
        n_missing += 1
        continue

    print(
        f"  [{n_plotted + 1:02d}/{n_stars}] {simbad_id}  "
        f"SpT={spectral_type}  V={v_mag:.2f}  field={field}  "
        f"N_visits={len(df_star)}"
    )

    plot_airmass_vs_mjd(
        df_star,
        simbad_id=simbad_id,
        spectral_type=spectral_type,
        v_mag=v_mag,
        field=field,
        save=True,
    )

    n_plotted += 1

print("=" * 70)
print(f"Done — plotted {n_plotted} stars, skipped {n_missing}.")